# 144 — QLoRA Fine-Tuning

**What you'll learn:**
- How 4-bit NF4 quantization reduces VRAM 8× compared to float32
- The difference between full fine-tuning, LoRA, and QLoRA
- Double quantization: quantizing the quantization constants
- How to read every parameter of BitsAndBytesConfig
- The LoRA rank tradeoff: r=4 lightweight vs r=64 high-capacity
- How to run a full QLoRA training loop and track VRAM usage

---
**Source:** `examples/144-qlora-finetuning/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Requires:** `OPENAI_API_KEY` — NOT needed. Part A is CPU-safe. Part B requires CUDA GPU (4GB+ VRAM).

In [ ]:
%pip install -q transformers peft trl datasets bitsandbytes torch

## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — Full Fine-Tuning vs LoRA vs QLoRA

| Method | Trainable Params | VRAM (7B model) | Quality Delta | When to use |
|--------|-----------------|-----------------|---------------|-------------|
| Full fine-tuning | 100% (7B) | ~56 GB (fp32) | Baseline | Unlimited compute |
| LoRA (bf16) | ~0.5% (~35M) | ~14 GB (bf16) | -1 to -3% | Good GPU (A100) |
| QLoRA (NF4) | ~0.5% (~35M) | ~3.5 GB (NF4) | -2 to -5% | Consumer GPU (T4) |

**NF4 (Normal Float 4)** places its 16 quantization bins according to the normal distribution. Neural network weights tend to follow a bell curve — so NF4 concentrates bins where weights are dense, unlike uniform int4 which wastes half its range on rarely-seen extremes.

```
int4 bins (uniform):  |-----|-----|-----|-----|-----|-----|-----|-----|
NF4 bins (normal):    |--|--|---|-----|-------|-----|---|--|--|
                      ← sparse extremes   dense center →
                      NF4 wastes less precision where weights actually live
```

### A2 — Double Quantization: Quantizing the Quantization Constants

In 4-bit quantization, each block of weights has a scale constant stored in float32 (32 bits). Double quantization quantizes those scale constants too, saving ~0.37 bits/param extra.

```
Standard NF4:
  weights → NF4 (4 bits)
  scale constants → float32 (32 bits per 64-weight block)
  overhead = 32/64 = 0.5 bits/param for scales

Double quantization:
  weights → NF4 (4 bits)
  scale constants → NF4 again (8 bits per 256-weight block)
  overhead ≈ 8/256 = 0.03 bits/param for scales

Savings: ~0.47 bits/param → for a 7B model ≈ 0.47 × 7×10⁹ / 8 ≈ 411 MB saved
```

In [ ]:
# A3 — VRAM estimation: given model size and quantization, estimate GPU memory
# No model is loaded — pure arithmetic.

BYTES_PER_DTYPE = {
    "float32": 4.0,
    "bfloat16": 2.0,
    "float16": 2.0,
    "int8": 1.0,
    "nf4": 0.5,       # 4 bits = 0.5 bytes
    "nf4_dq": 0.4625, # 4 bits + double quantization savings
}

def estimate_vram_gb(num_params: int, dtype: str, overhead_factor: float = 1.2) -> float:
    """
    Estimate VRAM usage for model weights.
    overhead_factor accounts for activations and optimizer states (during training).
    """
    bytes_per_param = BYTES_PER_DTYPE[dtype]
    model_bytes = num_params * bytes_per_param
    total_bytes = model_bytes * overhead_factor
    return round(total_bytes / 1e9, 2)

# VRAM table: common model sizes × dtypes
model_sizes = {
    "SmolLM2-135M": 135_000_000,
    "LLaMA2-7B":    7_000_000_000,
    "LLaMA2-13B":  13_000_000_000,
    "LLaMA2-70B":  70_000_000_000,
}

dtypes = ["float32", "bfloat16", "int8", "nf4", "nf4_dq"]

print(f"{'Model':<20}", end="")
for d in dtypes:
    print(f"  {d:<12}", end="")
print()
print("-" * (20 + 14 * len(dtypes)))

for name, params in model_sizes.items():
    print(f"{name:<20}", end="")
    for d in dtypes:
        gb = estimate_vram_gb(params, d, overhead_factor=1.0)  # weights only
        print(f"  {gb:<12.1f}", end="")
    print(" GB")

print()
print("Note: overhead_factor=1.2 adds ~20% for activations + gradients during training.")
print("QLoRA: backbone in NF4, but LoRA adapter weights + optimizer in float16.")

# Show QLoRA specific breakdown for 7B
params_7b = 7_000_000_000
lora_pct = 0.005  # ~0.5% trainable
print(f"\n--- QLoRA breakdown for 7B model ---")
backbone_gb = estimate_vram_gb(int(params_7b * (1 - lora_pct)), "nf4", 1.0)
adapter_gb  = estimate_vram_gb(int(params_7b * lora_pct), "float16", 1.5)  # + optimizer
print(f"Backbone (NF4):        {backbone_gb:.2f} GB")
print(f"Adapters (fp16 + opt): {adapter_gb:.2f} GB")
print(f"Total estimated:       {backbone_gb + adapter_gb:.2f} GB")
print(f"Fits on T4 (16GB)?     {'YES' if (backbone_gb + adapter_gb) < 16 else 'marginal'}")

In [ ]:
# A4 — BitsAndBytesConfig construction — show the config object attributes
# (bitsandbytes is installed but we don't load a model here)

from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("BitsAndBytesConfig attributes:")
print(f"  load_in_4bit            : {bnb_config.load_in_4bit}")
print(f"  bnb_4bit_quant_type     : {bnb_config.bnb_4bit_quant_type!r}")
print(f"  bnb_4bit_use_double_quant: {bnb_config.bnb_4bit_use_double_quant}")
print(f"  bnb_4bit_compute_dtype  : {bnb_config.bnb_4bit_compute_dtype}")

# Parameter guide
print()
print("Parameter guide:")
params = [
    ("load_in_4bit",             "True/False",            "Enables 4-bit weight storage on load"),
    ("bnb_4bit_quant_type",      '"nf4" or "fp4"',        "NF4 is better for normally-distributed weights"),
    ("bnb_4bit_use_double_quant","True/False",             "Saves ~0.4 bits/param extra via nested quantization"),
    ("bnb_4bit_compute_dtype",   "torch.float16/bfloat16","Upcast precision for matrix multiplications"),
]
for name, values, effect in params:
    print(f"  {name:<30} {values:<25}  {effect}")

In [ ]:
# A5 — LoRA trainable params formula: r*(d_in + d_out) per adapted layer
# Demonstrates how rank choice affects total trainable parameters.

def lora_params_per_layer(r: int, d_in: int, d_out: int) -> int:
    """A has shape (r, d_in), B has shape (d_out, r) → total = r*(d_in + d_out)"""
    return r * (d_in + d_out)

# SmolLM2-135M typical projection dimensions
d = 576   # SmolLM2-135M hidden size
n_layers = 30  # approximate

print("LoRA rank comparison (q_proj + v_proj, 30 layers)")
print(f"Projection dimension d={d}, layers={n_layers}\n")
print(f"{'Rank r':<10} {'Params/layer':<15} {'Total (q+v)':<15} {'vs d²':<12} {'Use case'}")
print("-" * 72)

use_cases = {4: "Lightweight adapter, fast training",
             8: "Balanced (default in most papers)",
             16: "Standard fine-tuning quality",
             32: "High capacity, approaching full FT",
             64: "Near full fine-tuning, heavy VRAM"}

for r in [4, 8, 16, 32, 64]:
    per_layer = lora_params_per_layer(r, d, d)
    total = per_layer * 2 * n_layers  # q_proj + v_proj
    full_ft_equiv = d * d * 2 * n_layers
    ratio = full_ft_equiv / total
    print(f"{r:<10} {per_layer:<15,} {total:<15,} {ratio:<12.1f}x  {use_cases[r]}")

print()
print("Rule of thumb: start with r=8. Double it if the model doesn't converge.")
print("Increasing r beyond 64 rarely helps and costs significant VRAM for adapters.")

In [ ]:
# A6 — Dataset format: instruction-input-output template
# Shows how raw Q/A pairs become training text with a prompt template.

SAMPLE_DATA = [
    ("What is gradient descent?",
     "Gradient descent is an optimization algorithm that iteratively adjusts "
     "parameters to minimize a loss function by moving in the direction of "
     "steepest decrease."),
    ("What is backpropagation?",
     "Backpropagation computes gradients of the loss with respect to each weight "
     "using the chain rule, propagating error from output to input layers."),
    ("What is overfitting?",
     "Overfitting occurs when a model learns the training data too well, including "
     "its noise, causing it to generalize poorly to new, unseen examples."),
]

def format_alpaca(instruction: str, response: str) -> str:
    """Alpaca-style instruction template."""
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n{response}"
    )

def format_qa(question: str, answer: str) -> str:
    """Simple Q/A template (used in src/workflow.py)."""
    return f"Q: {question}\nA: {answer}"

def format_chat(question: str, answer: str) -> str:
    """Chat template (closer to instruct model format)."""
    return (
        f"<|user|>\n{question}\n"
        f"<|assistant|>\n{answer}"
    )

print("Training data format comparison\n")
q, a = SAMPLE_DATA[0]

for fmt_name, fmt_fn in [("Q/A (simple)", format_qa),
                          ("Alpaca template", format_alpaca),
                          ("Chat template", format_chat)]:
    text = fmt_fn(q, a)
    tokens_approx = len(text.split())
    print(f"--- {fmt_name} (~{tokens_approx} tokens) ---")
    print(text[:200])
    print("..." if len(text) > 200 else "")
    print()

print("Tip: chat templates match the tokenizer's built-in chat format")
print("and tend to give better instruct-following after fine-tuning.")
print("The smolLM2-Instruct tokenizer has a built-in chat template.")

In [ ]:
# A7 — Count trainable parameters: mock model (no GPU, no model download)
# Mirrors tools.count_trainable() but works on plain Python dicts.

def count_trainable_mock(mock_params: list) -> dict:
    """
    mock_params: list of dicts with 'name', 'numel', 'requires_grad'
    Returns trainable/total counts and percentage.
    """
    total = sum(p['numel'] for p in mock_params)
    trainable = sum(p['numel'] for p in mock_params if p['requires_grad'])
    pct = 100.0 * trainable / total if total > 0 else 0.0
    return {'trainable': trainable, 'total': total, 'pct': round(pct, 4)}


# Mock SmolLM2-135M with QLoRA applied (r=8, q_proj + v_proj only)
# Real model: 135M params. With r=8 on q+v across 30 layers:
# LoRA params ≈ 2 * (8 * 576 + 576 * 8) * 30 ≈ 552,960
MOCK_MODEL = [
    # Frozen backbone (representative layers)
    {'name': 'embed_tokens.weight',           'numel': 49_152_000, 'requires_grad': False},
    {'name': 'layers.0.self_attn.q_proj.weight', 'numel': 331_776, 'requires_grad': False},
    {'name': 'layers.0.self_attn.v_proj.weight', 'numel': 331_776, 'requires_grad': False},
    {'name': 'layers.0.mlp.gate_proj.weight',    'numel': 884_736, 'requires_grad': False},
    # Trainable LoRA adapters (r=8)
    {'name': 'layers.0.self_attn.q_proj.lora_A.weight', 'numel': 4_608, 'requires_grad': True},
    {'name': 'layers.0.self_attn.q_proj.lora_B.weight', 'numel': 4_608, 'requires_grad': True},
    {'name': 'layers.0.self_attn.v_proj.lora_A.weight', 'numel': 4_608, 'requires_grad': True},
    {'name': 'layers.0.self_attn.v_proj.lora_B.weight', 'numel': 4_608, 'requires_grad': True},
]

stats = count_trainable_mock(MOCK_MODEL)
print(f"Trainable params  : {stats['trainable']:>12,}")
print(f"Total params      : {stats['total']:>12,}")
print(f"Trainable %       : {stats['pct']:>12.4f}%")
print()
print("In a real QLoRA run on SmolLM2-135M with r=8 (q+v, 30 layers):")
real_lora = 2 * (8 * 576 + 576 * 8) * 30
real_total = 135_000_000
print(f"  Trainable params ≈ {real_lora:,}  ({100*real_lora/real_total:.4f}% of {real_total/1e6:.0f}M total)")
print()
print("This is why QLoRA can run on a T4 GPU (16GB) — the optimizer states")
print("only cover ~500K params, not 135M.")

---
## Exercise 1: VRAM Savings Calculation

**Problem:** A Mistral-7B model has approximately 7.24 billion parameters.

Calculate:
1. VRAM required in bfloat16 (weights only, no overhead)
2. VRAM required in NF4 (weights only, no overhead)
3. VRAM required in NF4 with double quantization
4. The percentage savings from going bfloat16 → NF4 DQ

Use the formula: `VRAM_GB = num_params × bytes_per_param / 1e9`

| dtype | bytes/param |
|-------|------------|
| bfloat16 | 2.0 |
| nf4 | 0.5 |
| nf4 (double quant) | 0.4625 |

In [ ]:
# Exercise 1: VRAM savings for Mistral-7B going from bfloat16 to NF4

mistral_7b_params = 7_240_000_000  # 7.24B parameters

# TODO: calculate VRAM in GB for each dtype
vram_bf16 = None   # 2.0 bytes/param
vram_nf4  = None   # 0.5 bytes/param
vram_nf4_dq = None # 0.4625 bytes/param

# TODO: calculate percentage savings (bfloat16 as baseline)
savings_nf4    = None  # percent reduction
savings_nf4_dq = None  # percent reduction

# TODO: print a summary table
print("Mistral-7B VRAM requirements:")
# print(f"  bfloat16 : {vram_bf16:.2f} GB")
# print(f"  NF4      : {vram_nf4:.2f} GB  ({savings_nf4:.1f}% reduction)")
# print(f"  NF4 + DQ : {vram_nf4_dq:.2f} GB  ({savings_nf4_dq:.1f}% reduction)")

In [ ]:
# Exercise 1 — Answer Key

mistral_7b_params = 7_240_000_000

# VRAM in GB (weights only, no training overhead)
vram_bf16   = mistral_7b_params * 2.0 / 1e9      # 14.48 GB
vram_nf4    = mistral_7b_params * 0.5 / 1e9      #  3.62 GB
vram_nf4_dq = mistral_7b_params * 0.4625 / 1e9   #  3.35 GB

savings_nf4    = (1 - vram_nf4 / vram_bf16) * 100
savings_nf4_dq = (1 - vram_nf4_dq / vram_bf16) * 100

print("Mistral-7B VRAM requirements (weights only):")
print(f"  bfloat16 : {vram_bf16:.2f} GB  ← exceeds T4 (16GB) for training")
print(f"  NF4      : {vram_nf4:.2f} GB   ({savings_nf4:.1f}% reduction)")
print(f"  NF4 + DQ : {vram_nf4_dq:.2f} GB   ({savings_nf4_dq:.1f}% reduction)")
print()
# NF4 saves 75% of weight memory. Double quantization adds another ~7%.
# The key practical outcome: a 7B model that needs 14 GB in bf16
# fits in <4 GB as NF4, leaving room for activations and adapters.
print("Key insight: NF4 saves 75% of weight memory vs bfloat16.")
print("Double quantization saves an additional ~0.27 GB on a 7B model.")
print("Together, a 7B model fits comfortably on a T4 (16 GB) for fine-tuning.")

---
## Exercise 2: Alpaca Instruction Template

The training code in `src/workflow.py` uses a simple `Q: ... A: ...` format.
**Modify the formatter to use the Alpaca instruction template instead:**

```
### Instruction:
{instruction}

### Input:
{input}   ← optional context (can be empty)

### Response:
{response}
```

Your function should:
- Accept `instruction`, `input` (optional, default `""`), and `response`
- Return a single formatted string
- If `input` is empty, omit the `### Input:` section entirely

In [ ]:
# Exercise 2: Alpaca instruction template formatter

def format_alpaca_template(instruction: str, response: str, input: str = "") -> str:
    """
    Format a training example in Alpaca instruction format.
    If input is empty, omit the Input section.
    """
    # TODO: implement this function
    # Hint: use an if/else to conditionally include the Input section
    pass


# Test your implementation:
test_cases = [
    {
        "instruction": "Translate the following English text to French.",
        "input": "Hello, how are you?",
        "response": "Bonjour, comment allez-vous?",
    },
    {
        "instruction": "What is the capital of France?",
        "input": "",
        "response": "The capital of France is Paris.",
    },
]

for case in test_cases:
    result = format_alpaca_template(
        case["instruction"], case["response"], case.get("input", "")
    )
    print(result)
    print("---")

In [ ]:
# Exercise 2 — Answer Key

def format_alpaca_template(instruction: str, response: str, input: str = "") -> str:
    """
    Alpaca instruction template. Omits Input section when input is empty.
    This format was introduced in the Stanford Alpaca paper (2023) and is
    widely used for instruction fine-tuning datasets.
    """
    if input.strip():
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input}\n\n"
            f"### Response:\n{response}"
        )
    else:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        )

# Verify with both cases
test_cases = [
    ("Translate to French.", "Bonjour, comment allez-vous?", "Hello, how are you?"),
    ("What is the capital of France?", "The capital of France is Paris.", ""),
]

for instr, resp, inp in test_cases:
    result = format_alpaca_template(instr, resp, inp)
    print(result)
    print("---")

# Why Alpaca format matters:
# - It's consistent — the model learns to expect the ### markers
# - It separates instruction from context cleanly
# - Many open datasets (Alpaca-52K, Dolly-15K) use this exact format
# - Chat models often use a similar structure internally

---
## Part B — Full QLoRA Training Run

> **Requirements:**
> - CUDA GPU with at least **4 GB VRAM** (T4, RTX 3060, or better)
> - Free option: [Google Colab T4 runtime](https://colab.research.google.com) → Runtime > Change runtime type > T4 GPU
> - Cells will raise `EnvironmentError` on CPU-only machines — that is expected and intentional

**What Part B demonstrates:**
- Loading a model in 4-bit NF4 quantization
- Applying LoRA adapters with PEFT
- Running 30 training steps with `Trainer`
- Comparing VRAM before and after model load
- Printing trainable vs total parameter counts

In [ ]:
# B1 — GPU fail-fast check
# This cell MUST pass before running any Part B cells.

import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        "No CUDA GPU detected.\n"
        "Part B requires a GPU with 4GB+ VRAM.\n"
        "On Colab: Runtime > Change runtime type > T4 GPU (free tier).\n"
        "Re-run this cell after switching runtimes."
    )

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU detected  : {gpu_name}")
print(f"Total VRAM    : {vram_gb:.1f} GB")

if vram_gb < 4:
    raise EnvironmentError(
        f"GPU has only {vram_gb:.1f} GB VRAM. QLoRA requires at least 4 GB."
    )
print("GPU check passed. Ready for Part B.")

In [ ]:
# B2 — Full QLoRA training pipeline
# Loads model in 4-bit NF4, applies LoRA, trains 30 steps, reports metrics.

import sys, json, torch
sys.path.insert(0, '/content')  # Colab; adjust path if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "lora_rank": 8,
    "n_steps": 30,
    "vram_before_mb": 0.0,
    "vram_after_mb": 0.0,
    "param_stats": {},
    "final_loss": 0.0,
}

print(f"Model    : {state['model_name']}")
print(f"LoRA rank: r={state['lora_rank']}")
print(f"Steps    : {state['n_steps']}")
print()

result = workflow(state)

print()
print("=" * 50)
print("QLoRA Training Results")
print("=" * 50)
print(f"VRAM before model load : {result['vram_before_mb']:.0f} MB")
print(f"VRAM after model load  : {result['vram_after_mb']:.0f} MB")
print(f"VRAM delta             : {result['vram_after_mb'] - result['vram_before_mb']:.0f} MB")
print()
ps = result['param_stats']
print(f"Total parameters       : {ps.get('total', 0):,}")
print(f"Trainable parameters   : {ps.get('trainable', 0):,}")
print(f"Trainable %            : {ps.get('pct', 0):.4f}%")
print()
print(f"Final training loss    : {result['final_loss']}")
print()
print(f"Only {ps.get('pct', 0):.4f}% of parameters were updated — that's QLoRA's efficiency.")

In [ ]:
# B3 — Compare VRAM and final loss across LoRA ranks (live)
# Runs 3 short training experiments (r=4, r=8, r=16) and compares results.
# Each run takes ~1-2 minutes on a T4.

from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset
import torch

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
RANKS = [4, 8, 16]
N_STEPS = 15

print(f"Comparing r={RANKS} on {MODEL_NAME} ({N_STEPS} steps each)\n")

# Prepare shared dataset once
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

texts = ["Q: What is LoRA? A: LoRA adds low-rank adapter matrices to frozen weights."] * 20
enc = tokenizer(texts, truncation=True, max_length=64, padding="max_length", return_tensors="pt")
ds = Dataset.from_dict({k: v.tolist() for k, v in enc.items()})
ds = ds.map(lambda x: {"labels": x["input_ids"]})

results = []
for r in RANKS:
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    lora_cfg = LoraConfig(r=r, lora_alpha=r*2, target_modules=["q_proj","v_proj"],
                          lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(base, lora_cfg)

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    args = TrainingArguments(output_dir=f"/tmp/rank_{r}", max_steps=N_STEPS,
                             per_device_train_batch_size=2, learning_rate=2e-4,
                             logging_steps=N_STEPS, save_strategy="no", report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
    out = trainer.train()

    results.append({"r": r, "trainable": trainable, "pct": 100*trainable/total,
                    "loss": round(out.training_loss, 4)})
    del model, base
    torch.cuda.empty_cache()

print(f"\n{'Rank':<8} {'Trainable params':<20} {'% of total':<14} {'Final loss'}")
print("-" * 55)
for res in results:
    print(f"r={res['r']:<6} {res['trainable']:<20,} {res['pct']:<14.4f} {res['loss']}")

print()
print("Observation: higher rank = more params = typically lower loss,")
print("but with diminishing returns. r=8 is usually the sweet spot.")

### LoRA Alpha and the Scaling Factor

The LoRA update is: `ΔW = (alpha / r) × B × A`

The `alpha / r` ratio scales the adapter's contribution:

| Rule of thumb | Why |
|---|---|
| `alpha = r` | scaling factor = 1.0, adapter has full weight |
| `alpha = 2r` | scaling factor = 2.0, adapter has double weight |
| `alpha = r/2` | scaling factor = 0.5, conservative — good starting point |

**Most practitioners set `alpha = r` or `alpha = 2r`.** The defaults in PEFT are `lora_alpha=8` with `r=8` (ratio=1).
Only tune alpha after rank, target modules, and learning rate are stable.

### When NOT to Use QLoRA

QLoRA is not always the right choice:

| Scenario | Better choice | Reason |
|---|---|---|
| Have 80+ GB VRAM | Full fine-tuning | Max quality, no quantization noise |
| Need exact reproducibility | LoRA in fp16 | NF4 has stochastic rounding |
| Inference-only deployment | GGUF quantization | GGUF is optimized for inference, not training |
| Very small model (< 1B) | Full fine-tuning | The model is cheap enough to train fully |
| Production serving at scale | Merge LoRA + quantize separately | More control over quantization settings |

QLoRA shines for **training 7B–70B models on consumer or single-datacenter GPUs**.

In [ ]:
# Gradient checkpointing: trade compute for VRAM during training
# When enabled, activations are recomputed during backward pass instead of stored

def vram_with_gradient_checkpointing(base_gb: float, reduction_factor: float = 0.6) -> dict:
    """
    Gradient checkpointing reduces activation memory by ~60% at the cost of ~30% more compute.
    base_gb = VRAM needed without checkpointing.
    """
    activation_fraction = 0.35  # activations are ~35% of total VRAM during training
    activation_gb = base_gb * activation_fraction
    saved_gb = activation_gb * reduction_factor
    return {
        'without_gc': round(base_gb, 2),
        'with_gc': round(base_gb - saved_gb, 2),
        'saved_gb': round(saved_gb, 2),
        'extra_compute_pct': 30,
    }

for model_gb in [5.5, 8.0, 12.0, 18.0]:
    r = vram_with_gradient_checkpointing(model_gb)
    print(f'{model_gb:>5.1f} GB → {r["with_gc"]:>5.1f} GB with GC '
          f'(saves {r["saved_gb"]:.1f} GB, +{r["extra_compute_pct"]}% compute)')

print()
print('model.gradient_checkpointing_enable() is the one-line PEFT/transformers API.')
print('QLoRA + gradient checkpointing together: 7B model trains on ~6GB VRAM.')

### Sequence Packing for Throughput

Short training examples leave GPU memory unused. **Packing** concatenates multiple examples
into one context window, separated by EOS tokens:

```
Without packing (batch_size=4, max_len=512):
  [sample_1 (80 tokens) + 432 padding]
  [sample_2 (110 tokens) + 402 padding]
  [sample_3 (95 tokens) + 417 padding]
  [sample_4 (65 tokens) + 447 padding]
  Utilization: ~17%

With packing (max_len=512):
  [sample_1 + EOS + sample_2 + EOS + sample_3 + EOS + sample_4 + EOS + sample_5 + ...]
  Utilization: ~95%
```

TRL's `SFTTrainer` supports packing with `packing=True`. Use it when your dataset has
many short examples (< 256 tokens) and you want 3–5× throughput improvement.

---
## What's Next

- **QLoRA paper** (Dettmers et al. 2023, arxiv.org/abs/2305.14314) — original paper
- **PEFT library** (github.com/huggingface/peft) — LoRA, QLoRA, Prefix Tuning, and more
- **TRL SFTTrainer** (huggingface.co/docs/trl) — 10-line training with packing support
- **Example 145 — LoRA Architecture Ablation:** compare r=4 vs r=16 vs different target modules
- **Example 146 — ORPO Alignment:** align the fine-tuned model's preferences with ORPO

---
## What's Next

You've learned how QLoRA compresses a 7B model from 14 GB to ~3.5 GB while keeping fine-tuning quality close to full fine-tuning. Here's where to go deeper:

- **QLoRA paper**: Dettmers et al. (2023) — "QLoRA: Efficient Finetuning of Quantized LLMs" — https://arxiv.org/abs/2305.14314
- **LoRA paper**: Hu et al. (2021) — "LoRA: Low-Rank Adaptation of Large Language Models" — https://arxiv.org/abs/2106.09685
- **PEFT library docs**: https://huggingface.co/docs/peft — covers all adapter types (LoRA, Prefix Tuning, IA3)
- **bitsandbytes docs**: https://huggingface.co/docs/bitsandbytes — BitsAndBytesConfig reference
- **Example 145**: LoRA Architecture Ablation — systematically compare different rank and module configurations to see what actually matters
- **Example 146**: ORPO Alignment — once you have a fine-tuned model, use ORPO to align it to human preferences without a reference model